In [ ]:
!rm -rf PythonVehicleSimulator
!git clone https://github.com/cybergalactic/PythonVehicleSimulator.git
!cd /content/PythonVehicleSimulator && /usr/bin/python3 -m pip install -e .
# reinitializing python configuration after installing the package
import site
site.main()

Cloning into 'PythonVehicleSimulator'...
remote: Enumerating objects: 1425, done.
remote: Counting objects: 100% (275/275), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 1425 (delta 253), reused 214 (delta 214), pack-reused 1150 (from 2)
Receiving objects: 100% (1425/1425), 1.08 MiB | 5.69 MiB/s, done.
Resolving deltas: 100% (831/831), done.
Obtaining file:///content/PythonVehicleSimulator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for python_vehicle_simulator (pyproject.toml) ... done
  Created wheel for python_vehicle_simulator: filename=python_vehicle_simulator-1.0.1-0.editable-py3-none-any.whl size=2566 sha256=3e3136c1b5b75983d786f6f5516fd1a43da6f1480494372304fa49cb0a46af31
  Stored in directory: /tmp/pip-ephem-wheel-cache-g78d5itq/wheels/65/43/c7/aafc065f1742b4740af693ec5b7d2d

In the next block you can look into the function simulate if you want to have time dependent controller. You should first test your controller with constant reference first.

In [ ]:
import os
import sys
import webbrowser
import matplotlib.pyplot as plt
import numpy as np
import math
from python_vehicle_simulator.vehicles import remus100, otter
from python_vehicle_simulator.lib import (
    printVehicleinfo, attitudeEuler,plotControls,plotVehicleStates,plot3D
)

def sawtooth(x):
    # return (x + pi) % (2 * pi) - pi  # or equivalently   2*arctan(tan(x/2))
    return 2*np.arctan(np.tan(x/2))

def simulate(N, sampleTime, vehicle):

    DOF = 6                     # degrees of freedom
    t = 0                       # initial simulation time

    # Initial state vectors
    eta = np.array([0, 0, 0, 0, 0, 0], float)    # position/attitude, user editable
    nu = vehicle.nu                              # velocity, defined by vehicle class
    u_actual = vehicle.u_actual                  # actual inputs, defined by vehicle class

    # Initialization of table used to store the simulation data
    simData = np.empty( [0, 2*DOF + 2 * vehicle.dimU], float)

    # Simulator for-loop
    for i in range(0,N+1):
        t = i * sampleTime      # simulation time

        #-------------------------------------------------------------------------------------------------------------------------
        # If you want to have time dependent reference for the controller, do it here, otherwise comment
        # vehicle.yaw_d =
        # vehicle.u_d =
        #--------------------------------------------------------------------------------------------------------------------------

        # Vehicle specific control systems
        if (vehicle.controlMode == 'depthAutopilot'):
            u_control = vehicle.depthAutopilot(eta,nu,sampleTime)
        elif (vehicle.controlMode == 'headingAutopilot'):
            u_control = vehicle.headingAutopilot(eta,nu,sampleTime)
        elif (vehicle.controlMode == 'depthHeadingAutopilot'):
            u_control = vehicle.depthHeadingAutopilot(eta,nu,sampleTime)
        elif (vehicle.controlMode == 'DPcontrol'):
            u_control = vehicle.DPcontrol(eta,nu,sampleTime)
        elif (vehicle.controlMode == 'stepInput'):
            u_control = vehicle.stepInput(t)
        elif (vehicle.controlMode == 'speedDepthHeadingPiController'):
            u_control = vehicle.speedDepthHeadingPiController(eta,nu,sampleTime)
        elif (vehicle.controlMode =="speedheadingAutopilot"):
            u_control = vehicle.speedheadingAutopilot(eta,nu,sampleTime)

        # Store simulation data in simData
        signals = np.append( np.append( np.append(eta,nu),u_control), u_actual )
        simData = np.vstack( [simData, signals] )

        # Propagate vehicle and attitude dynamics
        [nu, u_actual]  = vehicle.dynamics(eta,nu,u_actual,u_control,sampleTime)
        eta = attitudeEuler(eta,nu,sampleTime)

    # Store simulation time vector
    simTime = np.arange(start=0, stop=t+sampleTime, step=sampleTime)[:, None]

    return(simTime,simData)

In the next block you need to fill the control functions

In [ ]:
class myOtter(otter):
  def __init__(
        self,
        controlSystem,
        u_d,
        psi_d,
        V_current,
        beta_current
    ):
    """
    Inputs:
        controlSystem: type of controller (string)
        u_d: desired speed (m/s)
        psi_d: desired heading angle (deg)
        V_c: current speed (m/s)
        beta_c: current direction (deg)
        tau_X: surge force, pilot input (N)
    """
    mycontrolSystem = str(controlSystem)
    super().__init__(controlSystem,0,V_current,beta_current,0)
    self.D2R = np.pi/180
    if mycontrolSystem == "speedheadingAutopilot":
            self.controlDescription = (
                "Speed, Heading autopilots, u_d = "
                + str(u_d)
                + " m/s"
                + ", psi_d = "
                + str(psi_d)
                + " deg"
                )
            self.u_d = u_d # to remember the value, constant by default
            self.a_d = 0 # desired acceleration, 0 by default
            self.psi_d = psi_d * self.D2R # to remember the value in rad
            self_r_d = 0 # desired angular velocity, 0 by default
            self.e_u_int = 0 # to rember the integral error of the speed
            self.e_psi_int = 0 # to rembmer the integral error of the headind
            self.controlMode = mycontrolSystem

  def speedheadingAutopilot(self, eta, nu, sampleTime):
        """
        u = speedheadingAutopilot(eta,nu,sampleTime) is a controller
        for automatic speed and heading control.

        Here you can implement a PI control for heading and a PID control for the heading

        """
        psi = eta[5]  # yaw angle
        r = nu[5]  # yaw rate
        r_d = self.r_d # desired yaw rate
        psi_d = self.psi_d # desired heading
        e_psi = sawtooth(psi - psi_d)  # yaw angle tracking error in rad
        e_r = r - r_d  # yaw rate tracking error

        u = nu[0] # the surge
        u_d= self.u_d # the desired surge
        e_u = u - u_d

        # there is no acceleration measurement unfortunately so no derivative term for the speed control

        # integrate the errors
        self.e_u_int += e_u * sampleTime
        self.e_psi_int += e_psi * sampleTime

        # PID gains
        Kpu, Kiu = 1.0, 1.0 # gains for speed
        Kpy, Kiy, Kdy = 1.0, 1.0, 1.0 # gain for heading

        # PID feedback controller with 3rd-order reference model
        tau_X =  # forward trust
        tau_N =  # turning momentum

        # control allocation, use self.Binv
        tau = np.array([tau_X, tau_N])  # tau = B * u_alloc
        u_alloc =  # u_alloc = inv(B) * tau

        # u_alloc = abs(n) * n --> n = sign(u_alloc) * sqrt(u_alloc)
        n_left =   # left thruster
        n_right =  # right thruster

        u_control = np.array([n_left, n_right], float) # left and right RPM

        return u_control


class myRemus1000(remus100):
    def __init__(self,
                controlSystem,
                z_d,
                psi_d,
                u_d,
                V_current,
                beta_current):
        """
        Inputs:
            controlSystem: type of controller (string)
            z_d: desired depth (m)
            u_d: desired speed (m/s)
            psi_d: desired heading angle (deg)
            V_c: current speed (m/s)
            beta_c: current direction (deg)
        """
        mycontrolSystem = str(controlSystem)
        super().__init__(controlSystem,0,0,0,V_current,beta_current)
        if mycontrolSystem == "speedDepthHeadingPiController":
            self.controlDescription = (
                "Speed, Depth and Heading autopilots, u_d = "
                + str(u_d)
                + " m/s"
                + ", z_d = "
                + str(z_d)
                + ", psi_d = "
                + str(psi_d)
                + " deg"
                )
            self.u_d = u_d # desired speed, constant but can be modified in the simulate function
            self.psi_d = psi_d * self.D2R # desired heading
            self.r_d = 0 # desired angular velocity, 0 by default
            self.z_d = z_d # desired depth
            self.e_u_int = 0 # integral terms
            self.e_psi_int = 0
            self.e_z_int = 0
            self.controlMode = mycontrolSystem

    def speedDepthHeadingPiController(self,eta,nu, sampleTime):
        """
        [delta_r, delta_s, n] = speedDepthHeadingPiController(eta,nu,sampleTime)
        simultaneously control the speed, depth and heading of the AUV.
        PID controller for heading and depth
        PI for speed

        Returns:

            u_control = [ delta_r   rudder angle (rad)
                         delta_s    stern plane angle (rad)
                         n          propeller revolution (rpm) ]

        """

        # get the reference point
        u_d = self.u_d # desired speed
        psi_d = self.psi_d # desired heading
        z_d = self.z_d # desired depth
        r_d = self.r_d # desired heading rate

        # get the state
        u = nu[0] # the speed
        psi = eta[5] # the heading
        z = eta[2] # depth

        pitch = eta[4] # pitch
        pitch_rate = nu[4] # pitch rate
        r = nu[5] # heading rate

        # errors
        e_u = u_d - u
        e_psi = sawtooth(psi_d - psi)
        e_r = r_d - r
        e_pitch_rate = 0-pitch_rate # reference is always 0
        e_pitch = 0-pitch # reference is always 0
        e_z =  z_d - z

        # integrate the errors
        self.e_u_int += e_u * sampleTime
        self.e_psi_int += e_psi * sampleTime
        self.e_z_int += e_z * sampleTime

        # PI or PID controllers
        delta_r =  # rudder angle
        delta_s =  # stern angle
        n = # propeller revolution

        u_control = np.array([delta_r, delta_s, n], float)

        return u_control

SyntaxError: invalid syntax (ipython-input-2752183932.py, line 65)

In [ ]:
### Simulation parameters ###
sampleTime = 0.02                   # sample time [seconds]
N = 10000                           # number of samples

# 3D plot and animation settings where browser = {firefox,chrome,safari,etc.}
numDataPoints = 50                  # number of 3D data points
FPS = 10                            # frames per second (animated GIF)
filename = '3D_animation.gif'       # data file for animated GIF

Please select the desired speed and the desired yaw when declaring the vehicle

In [ ]:
vehicle = myOtter("speedheadingAutopilot",1.,180.,0.,0) # u_d,yaw_d,current_speed,current_heading
printVehicleinfo(vehicle, sampleTime, N)
# Main simulation loop
[simTime, simData] = simulate(N, sampleTime, vehicle)

# 3D plots and animation
plotVehicleStates(simTime, simData, 1)
plotControls(simTime, simData, vehicle, 2)
#plot3D(simData, numDataPoints, FPS, filename, 3)

In [ ]:
vehicle = myRemus1000(controlSystem='speedDepthHeadingPiController', z_d=30, psi_d=50, u_d=1, V_current=0., beta_current=0)
printVehicleinfo(vehicle, sampleTime, N)

# Main simulation loop
[simTime, simData] = simulate(N, sampleTime, vehicle)

# 3D plots and animation
plotVehicleStates(simTime, simData, 1)
plotControls(simTime, simData, vehicle, 2)
plot3D(simData, numDataPoints, FPS, filename, 3)

In [ ]:
from IPython.display import Image
#Image(open('3D_animation.gif','rb').read())
